[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_13/MFM_ch13_anomaly_market/MFM_ch13_anomaly_market.ipynb)

# MFM_ch13_anomaly_market

**Modelling Financial Markets (MFM) — Chapter 13: Machine Learning in Finance**

Case study Dong, Li, Rapach and Zhou (2022), Anomalies and the Expected Market Return: recursive out-of-sample forecasts of the monthly market excess return (Mkt-RF) from 15 long-short anomaly returns of the Kenneth French Data Library (SMB, HML, RMW, CMA, momentum, short- and long-term reversal, extreme-decile spreads for accruals, net share issues, beta, variance, residual variance, E/P, CF/P, D/P). Methods: OLS on all predictors, elastic net (ENet), the simple combination of univariate forecasts (Comb) and the combination elastic net (C-ENet); training 1970-1979, validation 1980-1984, out-of-sample 1985 to the latest month. Campbell-Thompson out-of-sample R2 against the historical mean and the Clark-West test; chart of the cumulative gain in squared errors.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_13'
    os.makedirs(OUT_DIR, exist_ok=True)
    os.chdir(OUT_DIR)  # relative chart and CSV outputs persist in Drive
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


## Style and data

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts

SEED = 42


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Helper functions

In [ ]:
FRENCH_URL = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
FF_FILES = {'ff3': 'F-F_Research_Data_Factors_CSV.zip',
            'ff5': 'F-F_Research_Data_5_Factors_2x3_CSV.zip',
            # al doilea studiu de caz (Dong, Li, Rapach & Zhou 2022): factori si sortari univariate (decile, VW)
            'mom': 'F-F_Momentum_Factor_CSV.zip',
            'strev': 'F-F_ST_Reversal_Factor_CSV.zip',
            'ltrev': 'F-F_LT_Reversal_Factor_CSV.zip',
            **{k: f'Portfolios_Formed_on_{k}_CSV.zip' for k in ('AC', 'NI', 'BETA', 'VAR', 'RESVAR',
                                                                 'E-P', 'CF-P', 'D-P')}}


def french_factors(name='ff5'):
    """Factorii lunari FF3 / FF5 / Mom / reversal sau primul tabel (VW lunar) al unei sortari univariate
    (in zecimal, index = sfarsitul lunii) din Kenneth French Data Library."""
    import io
    import zipfile
    import urllib.request
    req = urllib.request.Request(FRENCH_URL + FF_FILES[name], headers={'User-Agent': 'Mozilla/5.0'})
    zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(req, timeout=120).read()))
    lines = zf.read(zf.namelist()[0]).decode('latin-1').splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(',') and len(l.split(',')) > 1)
    cols = [c.strip() for c in lines[i].split(',')[1:]]
    rows = []
    for l in lines[i + 1:]:
        parts = [x.strip() for x in l.split(',')]
        if len(parts[0]) != 6 or not parts[0].isdigit():      # sfarsitul tabelului lunar
            break
        rows.append([parts[0]] + [float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=['date'] + cols)
    df['date'] = pd.to_datetime(df['date'], format='%Y%m') + pd.offsets.MonthEnd(0)
    return df.set_index('date') / 100


START, FIRST_FC, OOS_START, PAPER_END = '1969-12', '1980-01-31', '1985-01-31', '2017-12-31'
# Tabelul 2 din lucrare (anomalii long-short, out-of-sample 1985-2017), R^2_OOS in %
DLRZ_R2 = {'OLS': -2513.86, 'ENet': 2.03, 'Comb': 0.89, 'C-ENet': 2.81}
LOW_MINUS_HIGH = ('AC', 'NI', 'BETA', 'VAR', 'RESVAR')
HIGH_MINUS_LOW = ('E-P', 'CF-P', 'D-P')
L1_RATIO = 0.5


def anomaly_panel():
    """Mkt-RF (tinta, luna t) si cele 15 anomalii long-short (luna t), lunar, in zecimal."""
    ff5 = french_factors('ff5')
    a = pd.DataFrame({'SMB': ff5['SMB'], 'HML': ff5['HML'], 'RMW': ff5['RMW'], 'CMA': ff5['CMA'],
                      'MOM': french_factors('mom').iloc[:, 0], 'STREV': french_factors('strev').iloc[:, 0],
                      'LTREV': french_factors('ltrev').iloc[:, 0]})
    for k in LOW_MINUS_HIGH:
        p = french_factors(k)
        a[k] = p['Lo 10'] - p['Hi 10']
    for k in HIGH_MINUS_LOW:
        p = french_factors(k)
        a[k.replace('-', '')] = p['Hi 10'] - p['Lo 10']
    return pd.concat([ff5['Mkt-RF'].rename('MKT'), a], axis=1).loc[START:].dropna()


def _aicc(n, rss, df):
    return n * np.log(rss / n) + 2 * (df + 1) + 2 * (df + 1) * (df + 2) / max(n - df - 2, 1)


def _enet_aicc(Z, y, positive=False, n_grid=40):
    """Elastic net (l1_ratio = 0.5) cu penalizarea aleasa prin AICc pe o grila logaritmica."""
    n = len(y)
    lam_max = np.max(np.abs((Z - Z.mean(0)).T @ (y - y.mean()))) / n / L1_RATIO
    best = None
    for lam in lam_max * np.logspace(0, -4, n_grid):
        m = ElasticNet(alpha=lam, l1_ratio=L1_RATIO, positive=positive, max_iter=20000).fit(Z, y)
        crit = _aicc(n, ((y - m.predict(Z)) ** 2).sum(), int((m.coef_ != 0).sum()))
        if best is None or crit < best[0]:
            best = (crit, m)
    return best[1]


def dlrz_forecasts(panel):
    """Prognoze recursive (fereastra extinsa) ale Mkt-RF: media istorica, 15 univariate, OLS, Comb, ENet, C-ENet."""
    y, X = panel['MKT'].values, panel.drop(columns='MKT').values
    dates, names = panel.index, list(panel.columns[1:])
    T, K = X.shape
    f0, oos = dates.get_loc(pd.Timestamp(FIRST_FC)), dates.get_loc(pd.Timestamp(OOS_START))
    hm, ols, enet, cenet = (np.full(T, np.nan) for _ in range(4))
    uni, nsel = np.full((T, K), np.nan), np.full(T, np.nan)
    for t in range(f0, T):                          # prognoza pentru luna t, cu date pina in t-1
        yy, xx = y[1:t], X[:t - 1]
        hm[t] = y[1:t].mean()
        for j in range(K):
            b, a = np.polyfit(xx[:, j], yy, 1)
            uni[t, j] = a + b * X[t - 1, j]
        Z = np.column_stack([np.ones(t - 1), xx])
        beta = np.linalg.lstsq(Z, yy, rcond=None)[0]
        ols[t] = beta[0] + X[t - 1] @ beta[1:]
        if t >= oos:
            sc = StandardScaler().fit(xx)
            enet[t] = _enet_aicc(sc.transform(xx), yy).predict(sc.transform(X[t - 1:t]))[0]
            sel = _enet_aicc(uni[f0:t], y[f0:t], positive=True).coef_ != 0
            nsel[t] = sel.sum()
            cenet[t] = uni[t, sel].mean() if sel.any() else hm[t]
    out = pd.DataFrame({'MKT': y, 'HM': hm, 'OLS': ols, 'ENet': enet, 'Comb': uni.mean(axis=1),
                        'C-ENet': cenet, 'nsel': nsel}, index=dates)
    return out.iloc[oos:], pd.DataFrame(uni, index=dates, columns=names).iloc[oos:]


def r2_oos(y, f, hm):
    """R^2_OOS Campbell-Thompson (2008), in %: 1 - SSE(model) / SSE(media istorica)."""
    return 100 * (1 - ((y - f) ** 2).sum() / ((y - hm) ** 2).sum())


def clark_west(y, f, hm):
    """Clark-West (2007): statistica t a diferentei ajustate de MSFE si p unilateral."""
    d = (y - hm) ** 2 - ((y - f) ** 2 - (hm - f) ** 2)
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    return t, 1 - stats.norm.cdf(t)


def summary(fc):
    res = {}
    for per, sl in (('1985-2017', slice(None, PAPER_END)), ('2018-end', slice('2018-01-31', None)),
                    ('1985-end', slice(None, None))):
        s = fc.loc[sl]
        res[per] = {m: dict(r2=round(r2_oos(s.MKT, s[m], s.HM), 3),
                            cw_t=round(clark_west(s.MKT, s[m], s.HM)[0], 3),
                            cw_p=round(clark_west(s.MKT, s[m], s.HM)[1], 3))
                    for m in ('OLS', 'ENet', 'Comb', 'C-ENet')}
        res[per]['T'] = len(s)
    res['cenet_selected_mean'] = round(float(fc.nsel.mean()), 3)
    res['cenet_none_share'] = round(float((fc.nsel == 0).mean()), 3)
    res['first'], res['last'] = f'{fc.index[0]:%Y-%m}', f'{fc.index[-1]:%Y-%m}'
    return res


## Chart

In [ ]:
def fig_dlrz_oos(fc, res, save=True):
    """Cistigul cumulat de erori patratice fata de media istorica (in %^2): Comb si C-ENet, 1985 - ultima luna."""
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    e0 = (fc.MKT - fc.HM) ** 2
    full = res['1985-end']
    for m, col, lab in (('Comb', MainBlue, 'Comb: mean of 15 univariate forecasts'),
                        ('C-ENet', IDAred, 'C-ENet: mean of the forecasts selected by elastic net')):
        gain = 1e4 * (e0 - (fc.MKT - fc[m]) ** 2).cumsum()
        ax.plot(fc.index, gain, color=col, lw=1.0,
                label=f'{lab} (R²OOS {full[m]["r2"]:.2f}%, {res["first"][:4]}-{res["last"][:4]})')
    ax.axhline(0, color=Navy, lw=0.5)
    ax.axvline(pd.Timestamp(PAPER_END), color=Navy, lw=0.7, ls='--')
    lo, hi = ax.get_ylim()
    ax.text(pd.Timestamp('2017-06-30'), lo + 0.04 * (hi - lo), "end of the paper's sample ",
            ha='right', va='bottom', fontsize=8, color='black')
    ax.set_ylabel('Cumulative gain in squared\nerrors vs. historical mean (%²)')
    ax.set_title(f'Anomaly returns forecasting the market excess return: out-of-sample '
                 f'{fc.index[0]:%b %Y} - {fc.index[-1]:%b %Y}', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=1, y=-0.13)
    plt.tight_layout()
    if save:
        save_fig('ch13_dlrz_oos')
    else:
        plt.close('all')


## Run

In [ ]:
panel = anomaly_panel()
fc, uni = dlrz_forecasts(panel)
res = summary(fc)
res['univariate_r2_1985_end'] = {k: round(r2_oos(fc.MKT, uni[k], fc.HM), 3) for k in uni}
res['paper_table2'] = DLRZ_R2
print(json.dumps(res, indent=1))
fig_dlrz_oos(fc, res)


![ch13_dlrz_oos](./ch13_dlrz_oos.png)

Output: `ch13_dlrz_oos.pdf`